# <center><span style='color:red'><b>EOF (without normalisation)</b></span></center>

## Imports
<hr style="border: solid 2px blue; margin-top: 1.5% ">

In [1]:
import xarray as xr
import numpy as np
from scipy.ndimage import uniform_filter1d
from eofs.xarray import Eof


In [2]:
# Open with xarray
data = xr.open_zarr('/gws/nopw/j04/co2clim/datasets/GLORYS/GLOBAL_MULTIYEAR_PHY_001_030/Subset_NorthAtlantic_regrid1x1/cmems_mod_glo_phy_my_0.083deg_P1D-m_mlotst_100.00W-20.00E_0.00N-80.00N_1993-01-01-2021-06-30.zarr').load()

# Print a summary of dataset
print(data)

<xarray.Dataset> Size: 408MB
Dimensions:    (latitude: 81, longitude: 121, time: 10408)
Coordinates:
  * latitude   (latitude) float64 648B 0.0 1.0 2.0 3.0 ... 77.0 78.0 79.0 80.0
  * longitude  (longitude) float64 968B -100.0 -99.0 -98.0 ... 18.0 19.0 20.0
  * time       (time) datetime64[ns] 83kB 1993-01-01 1993-01-02 ... 2021-06-30
Data variables:
    mlotst     (time, latitude, longitude) float32 408MB 10.53 10.53 ... nan nan


## Clean and process data
<hr style="border: solid 2px blue; margin-top: 1.5% ">

In [3]:
# ----------------------------------
# Required variable
# ----------------------------------
varname = "mlotst"

# ----------------------------------
# Ensure required dimensions
# ----------------------------------
required_dims = {"time", "latitude", "longitude"}
if not required_dims.issubset(data[varname].dims):
    raise ValueError(
        f"{varname} must have dimensions {required_dims}, "
        f"found {data[varname].dims}")

# ----------------------------------
# Enforce consistent dimension order
# ----------------------------------
data = data.transpose("time", "latitude", "longitude")

# ----------------------------------
# Select only full years (1993-2020)
# ----------------------------------
data = data.sel(time=slice("1993-01-01", "2020-12-31"))

# ----------------------------------
# Remove leap days (Feb 29)
# ----------------------------------
is_feb29 = (data.time.dt.month == 2) & (data.time.dt.day == 29)
data = data.sel(time=~is_feb29)

# ----------------------------------
# 5. Resample to daily frequency
# ----------------------------------
# Ensures strictly daily time coordinate; missing days become NaN
data = data[varname].resample(time="1D").asfreq()

# ----------------------------------
# Sanity checks (optional)
# ----------------------------------
#print("Time range:", data.time.min().values, "to", data.time.max().values)
#print("Time step (should be 1 day):", np.diff(data.time.values).astype("timedelta64[D]").astype(int)[:10])
#print("Fraction of NaNs in dataset:", np.isnan(data).mean().values)

## Compute climatology and anomalies
<hr style="border: solid 2px blue; margin-top: 1.5% ">

In [4]:
# ----------------------------------
# Parameters for climatology/anomalies
# ----------------------------------
climo_smooth_days = 1   # Number of days for smoothing the daily climatology
# Set >1 if you want to smooth climatology in time (ERA often used 1 or 5)

# ----------------------------------
# 1. Compute daily climatology
# ----------------------------------
# Group by day-of-year (1–365) and compute mean across years
climatology = data.groupby("time.dayofyear").mean(dim="time", skipna=True)

# ----------------------------------
# 2. Optional smoothing
# ----------------------------------
if climo_smooth_days > 1:
    # Apply 1D smoothing along the day-of-year dimension
    climatology = xr.DataArray(uniform_filter1d(climatology, size=climo_smooth_days, axis=0, mode="wrap"), coords=climatology.coords, dims=climatology.dims)

# ----------------------------------
# 3. Compute anomalies
# ----------------------------------
# Expand climatology to full time dimension and subtract
# Use .groupby so the day-of-year aligns with the full time series
anomalies = data.groupby("time.dayofyear") - climatology

# ----------------------------------
# 4. Quick sanity checks
# ----------------------------------
print("Climatology shape:", climatology.shape)
print("Anomalies shape:", anomalies.shape)
print("Example climatology values (day 1, first lat/lon):", climatology.isel(dayofyear=0, latitude=0, longitude=0).values)
print("Example anomaly values (first day, first lat/lon):", anomalies.isel(time=0, latitude=0, longitude=0).values)

# ----------------------------------
# Output
# ----------------------------------
# climatology(dayofyear, latitude, longitude)
# anomalies(time, latitude, longitude)


Climatology shape: (366, 81, 121)
Anomalies shape: (10227, 81, 121)
Example climatology values (day 1, first lat/lon): 11.275499
Example anomaly values (first day, first lat/lon): -0.7466135


## Seasonal subsetting
<hr style="border: solid 2px blue; margin-top: 1.5% ">

In [5]:
# ----------------------------------
# Keep full anomaly dataset
# ----------------------------------
full_anomalies = anomalies

# ----------------------------------
# Seasonal subsetting of anomalies
# ----------------------------------
DJF_anomalies = anomalies.sel(time=anomalies.time.dt.month.isin([12, 1, 2])).sortby("time")
MAM_anomalies = anomalies.sel(time=anomalies.time.dt.month.isin([3, 4, 5])).sortby("time")
JJA_anomalies = anomalies.sel(time=anomalies.time.dt.month.isin([6, 7, 8])).sortby("time")
SON_anomalies = anomalies.sel(time=anomalies.time.dt.month.isin([9, 10, 11])).sortby("time")

# ----------------------------------
# Check
# ----------------------------------
#print("DJF:", DJF_anomalies.shape)
#print("MAM:", MAM_anomalies.shape)
#print("JJA:", JJA_anomalies.shape)
#print("SON:", SON_anomalies.shape)
#print("Full:", full_anomalies.shape)


## Compute EOFs
<hr style="border: solid 2px blue; margin-top: 1.5% ">

#### Handle NaNs

In [9]:
# ----------------------------------
# Preprocess seasonal anomalies: drop fully-NaN spatial points
# ----------------------------------
def drop_fully_nan_points(anom_data, name="Dataset"):
    """
    Drop spatial points that are NaN for all times.
    Prints summary of remaining grid and checks that some points remain.
    
    Parameters:
        anom_data: xarray.DataArray (time, lat, lon)
        name: string, dataset name for printout
        
    Returns:
        cleaned_data: xarray.DataArray with only valid points
    """
    nt, nlat, nlon = anom_data.shape
    
    # Flatten spatial dimensions
    arr_flat = anom_data.values.reshape(nt, nlat*nlon)
    
    # Identify columns that are NOT fully NaN
    valid_cols = ~np.all(np.isnan(arr_flat), axis=0)
    
    n_removed = np.sum(~valid_cols)
    n_remaining = np.sum(valid_cols)
    
    #print(f"{name}:")
    #print(f"  Total fully NaN grid points removed: {n_removed} / {nlat*nlon}")
    #print(f"  Remaining valid points: {n_remaining} / {nlat*nlon}")
    
    # Check if any valid points remain
    if n_remaining == 0:
        raise ValueError(f"All spatial points in {name} are NaN! Cannot continue.")
    
    # Map 1D valid_cols back to lat/lon indices
    valid_idx = np.array(np.unravel_index(np.where(valid_cols)[0], (nlat, nlon))).T
    lat_idx = np.unique(valid_idx[:,0])
    lon_idx = np.unique(valid_idx[:,1])
    
    # Subset the DataArray to only valid lat/lon
    cleaned_data = anom_data.sel(latitude=anom_data.latitude[lat_idx], longitude=anom_data.longitude[lon_idx])
    
    return cleaned_data

# ----------------------------------
# Apply to each seasonal dataset
# ----------------------------------
DJF_anomalies_clean = drop_fully_nan_points(DJF_anomalies, "DJF anomalies")
MAM_anomalies_clean = drop_fully_nan_points(MAM_anomalies, "MAM anomalies")
JJA_anomalies_clean = drop_fully_nan_points(JJA_anomalies, "JJA anomalies")
SON_anomalies_clean = drop_fully_nan_points(SON_anomalies, "SON anomalies")
full_anomalies_clean = drop_fully_nan_points(full_anomalies, "Full anomalies")


#### Function

In [12]:
def compute_eofs_robust(anom_data, n_eof, name="Dataset"):
    """
    Robust EOF solver using xarray DataArray
    """
    # 1. Drop fully NaN spatial points
    valid_lat = ~np.all(np.isnan(anom_data), axis=(0,2))
    valid_lon = ~np.all(np.isnan(anom_data), axis=(0,1))
    data_clean = anom_data.sel(latitude=anom_data.latitude[valid_lat],
                               longitude=anom_data.longitude[valid_lon])

    nt, nlat, nlon = data_clean.shape

    if nt == 0 or nlat == 0 or nlon == 0:
        raise ValueError(f"{name}: No valid data left after dropping fully NaN points.")

    # 2. Cosine-latitude weights for remaining lats
    lat_vals = data_clean.latitude.values   # convert to NumPy
    weights = np.sqrt(np.cos(np.radians(lat_vals)))
    weights_2d = np.broadcast_to(weights[:, np.newaxis], (nlat, nlon))

    # 3. EOF solver
    solver = Eof(data_clean, weights=weights_2d)

    # 4. Extract EOFs, PCs, variance
    eof_patterns = solver.eofs(neofs=n_eof, eofscaling=1)
    pcs = solver.pcs(npcs=n_eof, pcscaling=0)
    explained_var = solver.varianceFraction(neigs=n_eof)

    return eof_patterns, pcs, explained_var


#### Solver

In [13]:
DJF_eof, DJF_pcs, DJF_var = compute_eofs_robust(DJF_anomalies_clean, n_eof_season, "DJF")
MAM_eof, MAM_pcs, MAM_var = compute_eofs_robust(MAM_anomalies_clean, n_eof_season, "MAM")
JJA_eof, JJA_pcs, JJA_var = compute_eofs_robust(JJA_anomalies_clean, n_eof_season, "JJA")
SON_eof, SON_pcs, SON_var = compute_eofs_robust(SON_anomalies_clean, n_eof_season, "SON")

full_eof, full_pcs, full_var = compute_eofs_robust(full_anomalies_clean, n_eof_full, "Full")


ValueError: all input data is missing